**Importar librerías**

In [ ]:
import os
from pathlib import Path
import pandas as pd
import SimpleITK as sitk
from totalsegmentator.python_api import totalsegmentator
import dicom2nifti
import numpy as np
import nibabel as nib
from skimage import measure
import cv2
from reorient_nii import reorient

**Cargar directorio de estudios**

In [12]:
dataset_path = Path("CT_dicoms")
studies = []
conditions = ["ADX","IMH","PAU", "NC"]
for condition in conditions:
    route = dataset_path / condition
    if route.exists():
        for path in route.iterdir():
            studies.append({
                    "Nombre_Estudio": path.name, 
                    "Etiqueta": condition
                })
dataset = pd.DataFrame(studies)
dataset.to_csv('dataset_index.csv')

**Cargar dicoms y convertirlos a NIfTI**

In [14]:
for index, row in dataset.iterrows():
    condition = row["Etiqueta"]
    file_name = row["Nombre_Estudio"]
    input_path = dataset_path / condition / file_name
    output_path = f"CT_dataset/{condition}/{file_name}.nii.gz"
    if Path(output_path).is_file() != True:
        try:
            print(f"Procesando {input_path}")
            dicom2nifti.dicom_series_to_nifti(input_path, output_path, reorient_nifti=True)
            print(f"¡Conversión exitosa! Revisa la carpeta: {output_path}") 
        except Exception as e:
            print(f"Ocurrió un error durante la conversión: {e}")
            continue
    else:
        print(f"El archivo {output_path} ya existe.")

El archivo CT_dataset/ADX/Mario_D'Oria_ADX_ITA_GE_0.625.nii.gz ya existe.
El archivo CT_dataset/ADX/Mario_D'Oria_ADX_ITA_PHILIPS_1.000.nii.gz ya existe.
El archivo CT_dataset/ADX/Mario_D'Oria_ADX_ITA_PHILIPS_2.000.nii.gz ya existe.
El archivo CT_dataset/ADX/Daniele_Morosetti_ADX_ITA_GE_2.5.nii.gz ya existe.
El archivo CT_dataset/ADX/Daniele_Morosetti_ADX_ITA_GE_2.5 1.nii.gz ya existe.
El archivo CT_dataset/ADX/Daniele_Morosetti_ADX_ITA_PHILIPS_2.nii.gz ya existe.
El archivo CT_dataset/ADX/Marta_Pignataro_ADX_ITA_GE_1.25.nii.gz ya existe.
El archivo CT_dataset/ADX/Marta_Pignataro_ADX_ITA_GE_2.5.nii.gz ya existe.
El archivo CT_dataset/IMH/Mario_D'Oria_IMH_ITA_GE_1.250.nii.gz ya existe.
El archivo CT_dataset/IMH/Marta_Pignataro_IMA_ITA_GE_2.5.nii.gz ya existe.
El archivo CT_dataset/IMH/Daniele_Morosetti_IMH_ITA_GE_1.25.nii.gz ya existe.
El archivo CT_dataset/IMH/Daniele_Morosetti_IMH_ITA_GE_5.nii.gz ya existe.
El archivo CT_dataset/NC/NORMAL.nii.gz ya existe.
El archivo CT_dataset/NC/Mart

Missing slices (slice count mismatch between timepoint 0 and 1)
---------------------------------------------------------
(512, 512, 294)
(512, 512, 9)
---------------------------------------------------------


Ocurrió un error durante la conversión: MISSING_DICOM_FILES


**Extraer máscaras (Segmentación)**

In [10]:
#Extracción de mascaras
print("Iniciando segmentaciones")
for index, row in dataset.iterrows():
    condition = row["Etiqueta"]
    file_name = row["Nombre_Estudio"]
    input_path = f"CT_dataset/{condition}/{file_name}.nii.gz"
    print(input_path)
    output_path = f"CT_mask/{condition}/{file_name}/"
    mask_route = f"CT_mask/{condition}/{file_name}/aorta.nii.gz"
    if Path(mask_route).is_file() != True:
        totalsegmentator(input=input_path, output=output_path, roi_subset=["aorta"], fast=False, device="cpu" )
    else:
        print(f"La máscara {mask_route} ya existe.")  

print("Segementaciones completadas")

Iniciando segmentaciones
CT_dataset/ADX/Mario_D'Oria_ADX_ITA_GE_0.625.nii.gz
La máscara CT_mask/ADX/Mario_D'Oria_ADX_ITA_GE_0.625/aorta.nii.gz ya existe.
CT_dataset/ADX/Mario_D'Oria_ADX_ITA_PHILIPS_1.000.nii.gz
La máscara CT_mask/ADX/Mario_D'Oria_ADX_ITA_PHILIPS_1.000/aorta.nii.gz ya existe.
CT_dataset/ADX/Mario_D'Oria_ADX_ITA_PHILIPS_2.000.nii.gz
La máscara CT_mask/ADX/Mario_D'Oria_ADX_ITA_PHILIPS_2.000/aorta.nii.gz ya existe.
CT_dataset/ADX/Daniele_Morosetti_ADX_ITA_GE_2.5.nii.gz
La máscara CT_mask/ADX/Daniele_Morosetti_ADX_ITA_GE_2.5/aorta.nii.gz ya existe.
CT_dataset/ADX/Daniele_Morosetti_ADX_ITA_GE_2.5 1.nii.gz
La máscara CT_mask/ADX/Daniele_Morosetti_ADX_ITA_GE_2.5 1/aorta.nii.gz ya existe.
Segementaciones completadas


In [2]:
dataset_path = Path("CT_mask")
studies = []
conditions = ["ADX","IMH","PAU", "NC"]
for condition in conditions:
    route = dataset_path / condition
    if route.exists():
        for path in route.iterdir():
            studies.append({
                    "Nombre_Estudio": path.name, 
                    "Etiqueta": condition
                })
dataset = pd.DataFrame(studies)
dataset.to_csv('dataset_index_all.csv')

In [ ]:
def features_extractor(path, file_name, pathology):
    """
    En esta sección es necesario automatizar la dirección de la ruta del archivo DICOM,
    para ello se deben recibir los archivos del código de Linux.
    """

    condition = pathology

    image = nib.load(path)
    header = image.header
    voxel_dims = header.get_zooms()
    image = reorient(image,'SPR')
    data_array = image.get_fdata()

    #Corte de cara de salida y entrada
    mask = sitk.GetImageFromArray(data_array)
    mask_limpia_3d = sitk.GetArrayFromImage(mask)

    mask_limpia_3d = np.copy(mask_limpia_3d) 
    total_slices = mask_limpia_3d.shape[0]
    
    slices_offset_iliacas = 10
    z_activos = np.where(np.any(mask_limpia_3d, axis=(1, 2)))[0]
    if len(z_activos) == 0:
        print("La máscara está vacía.")

    z_min = z_activos[0]
    z_max = z_activos[-1]
    altura_real = z_max - z_min
    umbral_55 = z_min + int(altura_real * 0.55)

    # =========================================================================
    # FASE 1: Limpieza de Ilíacas (Barrido Inverso)
    # =========================================================================
    z_corte_final_inferior = z_min 
    for z in range(umbral_55, z_min - 1, -1):
        slice_actual = mask_limpia_3d[z, :, :].astype(np.uint8)
        if np.any(slice_actual):
            num_labels, _, _, _ = cv2.connectedComponentsWithStats(slice_actual, connectivity=8)
            if num_labels >= 3:
                z_corte_final_inferior = z + slices_offset_iliacas
                break

    if z_corte_final_inferior > total_slices: z_corte_final_inferior = total_slices
    mask_limpia_3d[0 : z_corte_final_inferior + 1, :, :] = 0
    z_inicio_aorta = z_corte_final_inferior + 1
    

    """# --- VARIABLES CONFIGURABLES ---
    slices_iniciales_a_borrar = 10     # Slices a borrar de la salida
    slices_superiores_a_modificar = 10 # Slices a modificar de la entarda

    # 1. Borrar las slices de la salida
    z_con_mascara = np.where(np.any(mask_limpia_3d, axis=(1, 2)))[0]

    indices_iniciales_a_borrar = z_con_mascara[:slices_iniciales_a_borrar]

    for z in indices_iniciales_a_borrar:
        mask_limpia_3d[z, :, :] = 0


    # 2. Modificar slices de la entrada
    umbral_55 = int(total_slices * 0.55)
    slices_modificadas = 0

    for z in range(umbral_55, total_slices):
        
        if slices_modificadas >= slices_superiores_a_modificar:
            break 
            
        slice_actual = mask_limpia_3d[z, :, :].astype(np.uint8)
        
        if np.any(slice_actual):
            

            num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(slice_actual, connectivity=8)
            
            if num_labels >= 3:
                min_y = float('inf')
                label_a_eliminar = -1
                
                # Buscar el cuerpo más próximo al cero en el eje Y
                for i in range(1, num_labels):
                    centroid_y = centroids[i][1] # El índice 1 corresponde al eje Y
                    
                    if centroid_y < min_y:
                        min_y = centroid_y
                        label_a_eliminar = i
                
                # Si encontramos el cuerpo, lo rellenamos con ceros
                if label_a_eliminar != -1:
                    # Todo lo que corresponda a esa etiqueta se vuelve 0
                    slice_actual[labels == label_a_eliminar] = 0
                    
                    # Actualizamos la matriz 3D
                    mask_limpia_3d[z, :, :] = slice_actual
                    
                    # Aumentamos el contador de slices modificadas
                    slices_modificadas += 1
    """
    mask_limpia_sitk = sitk.GetImageFromArray(mask_limpia_3d)
    mask_limpia_sitk.CopyInformation(mask)

    #Extrer regiones

    def segmentar_regiones_aorta(mask_array):
        mask_sitk = mask_array
        mask_3d = sitk.GetArrayFromImage(mask_sitk)
        
        # =========================================================================
        # CORTE HORIZONTAL (Eje Z) - Límite entre Arcos y Descenso
        # =========================================================================
        # Encontrar la longitud real en Z
        z_activos = np.where(np.any(mask_3d, axis=(1, 2)))[0]
        if len(z_activos) == 0:
            print("La máscara está vacía.")
            return
            
        z_min = z_activos[0]
        z_max = z_activos[-1]
        altura_real = z_max - z_min
        umbral_55 = z_min + int(altura_real * 0.55)

        # Barrido desde el 55% hacia arriba buscando 2 cuerpos
        z_corte_horizontal = umbral_55 # Valor por defecto si falla
        for z in range(umbral_55, z_max + 1):
            slice_actual = mask_3d[z, :, :].astype(np.uint8)
            if np.any(slice_actual):
                num_labels, _, _, _ = cv2.connectedComponentsWithStats(slice_actual, connectivity=8)
                # num_labels == 3 significa: 1 Fondo + 2 Cuerpos (Ascendente y Descendente)
                if num_labels == 3:
                    z_corte_horizontal = z
                    break

        # =========================================================================
        # CORTE VERTICAL (Eje Y) - Límite entre Arco Ascendente y Descendente
        # =========================================================================
        # Barrido en el eje Y (Coronal/Sagital). axis=1 en (Z, Y, X) es el eje Y.
        # Evaluamos en qué coordenadas Y hay información de la aorta.
        y_activos = np.where(np.any(mask_3d, axis=(0, 2)))[0]
        
        y_min = y_activos[0]  # Primer barrido de 0 a len
        y_max = y_activos[-1] # Segundo barrido de len a 0
        
        # El corte se hace exactamente en la mitad de la caja delimitadora en Y
        y_corte_vertical = (y_min + y_max) // 2

        # =========================================================================
        # GENERACIÓN DE LAS 3 MÁSCARAS
        # =========================================================================
        # Crear arrays vacíos con la misma forma
        mask_descenso = np.zeros_like(mask_3d)
        mask_arco_asc = np.zeros_like(mask_3d)
        mask_arco_desc = np.zeros_like(mask_3d)

        # 1. DESCENSO: Todo lo que esté por debajo del corte horizontal Z
        mask_descenso[:z_corte_horizontal, :, :] = mask_3d[:z_corte_horizontal, :, :]

        # 2 y 3. ARCOS: Todo lo que esté por encima del corte Z, dividido por el corte Y
        bloque_superior = mask_3d[z_corte_horizontal:, :, :]
        
        # Extraemos el bloque superior y aplicamos la "guillotina" en Y
        bloque_asc = np.copy(bloque_superior)
        bloque_desc = np.copy(bloque_superior)
        
        # NOTA DE ORIENTACIÓN: Dependiendo de tu tomografía, el Y menor puede ser anterior o posterior.
        # Por estándar médico, Y=0 suele ser la parte anterior (pecho) y el Y_max la posterior (espalda).
        # Por lo tanto, el Arco Ascendente está en los Y menores al corte.
        
        # Al ascendente le borramos la parte de la espalda (Y >= corte)
        bloque_asc[:, y_corte_vertical:, :] = 0 
        
        # Al descendente le borramos la parte del pecho (Y < corte)
        bloque_desc[:, :y_corte_vertical, :] = 0 
        
        # Guardamos los bloques en sus máscaras finales
        mask_arco_asc[z_corte_horizontal:, :, :] = bloque_asc
        mask_arco_desc[z_corte_horizontal:, :, :] = bloque_desc


        mask_descenso = sitk.GetImageFromArray(mask_descenso)
        mask_descenso.CopyInformation(mask_sitk)
        mask_arco_asc = sitk.GetImageFromArray(mask_arco_asc)
        mask_arco_asc.CopyInformation(mask_sitk)
        mask_arco_desc = sitk.GetImageFromArray(mask_arco_desc)
        mask_arco_desc.CopyInformation(mask_sitk)
        
        return  mask_descenso, mask_arco_asc, mask_arco_desc

    mask_descenso, mask_arco_asc, mask_arco_desc = segmentar_regiones_aorta(mask_limpia_sitk)

    #Codigo de volumen a base del voxel , por region y completo

    def calcular_volumen(image, nombre_estructura, valor_etiqueta=1):
        # Cargar datos
        mask_array = sitk.GetArrayFromImage(image)
        
        # Calcular volumen
        vol_voxel_mm3 = voxel_dims[0] * voxel_dims[1] * voxel_dims[2]
        num_voxeles = np.sum(mask_array == valor_etiqueta)
        vol_total_mm3 = num_voxeles * vol_voxel_mm3
        vol_total_cm3 = vol_total_mm3 / 1000.0
        
        return vol_total_cm3

    def calculate_surface_area(image, voxel_size):
        """
        Calcula el área de superficie de una máscara 3D en mm^2.
        
        Args:
            mask_array: Array binario (numpy) de la máscara.
            voxel_size: Lista o tuple con [spacing_x, spacing_y, spacing_z].
        """
        mask_array = sitk.GetArrayFromImage(image)
        # 1. Extraer la superficie usando Marching Cubes
        # El parámetro 'level' suele ser 0.5 para máscaras binarias (0 y 1)
        verts, faces, normals, values = measure.marching_cubes(mask_array, level=0.5, spacing=tuple(voxel_size))
        
        # 2. Calcular el área de la superficie
        surface_area = measure.mesh_surface_area(verts, faces)
        
        return surface_area

    def calcular_areas_salida(image, nombre_estructura, tipo_cara='primera', valor_etiqueta=1):
        mask_array = sitk.GetArrayFromImage(image)

        coordenadas = np.where(mask_array == valor_etiqueta)
        
        if len(coordenadas[0]) == 0:
            print(f" La máscara de {nombre_estructura} está completamente vacía.")
            return None

        indices_z = coordenadas[2]
        
        if tipo_cara == 'primera':
            corte_objetivo = np.min(indices_z)
            texto_cara = "PRIMERA cara (Corte Z más bajo)"
        elif tipo_cara == 'ultima':
            corte_objetivo = np.max(indices_z)
            texto_cara = "ÚLTIMA cara (Corte Z más alto)"
        else:
            return None

        #Extraccion del el corte y contar los vóxeles
        corte_2d = mask_array[:, :, corte_objetivo]
        num_voxeles_cara = int(np.sum(corte_2d == valor_etiqueta))

        # CONVERSIÓN A MILÍMETROS (ÁREA)
        area_un_pixel_mm2 = voxel_dims[0] * voxel_dims[1]
        area_total_mm2 = num_voxeles_cara * area_un_pixel_mm2

        return area_total_mm2

    def calcular_metricas_transversales(image, nombre_estructura, voxel_dims, valor_etiqueta=1):
        """
        Calcula el diámetro mayor, menor y la excentricidad de la aorta slice por slice.
        Retorna un diccionario con los valores máximos, mínimos, promedios y std.
        """
        mask_array = sitk.GetArrayFromImage(image)
        
        # En SimpleITK el array extraído siempre tiene la forma (Z, Y, X).
        # Por lo tanto, el eje Z (los slices) es el índice 0.
        total_slices = mask_array.shape[0]
        
        diametros_mayores = []
        diametros_menores = []
        excentricidades = []
        
        # Para la conversión a mm, usamos la resolución del eje X (asumiendo que X e Y 
        # son isotrópicos/iguales, lo cual es el estándar en tomografías axiales).
        pixel_spacing_mm = voxel_dims[0]

        for z in range(total_slices):
            slice_2d = mask_array[z, :, :]
            
            # Procesar solo si hay aorta en este slice
            if np.any(slice_2d == valor_etiqueta):
                
                # Etiquetar las regiones del slice (por si queda algún píxel flotante)
                labels = measure.label(slice_2d == valor_etiqueta)
                props = measure.regionprops(labels)
                
                if props:
                    # Encontrar el objeto más grande del slice (la luz de la aorta)
                    prop_aorta = max(props, key=lambda item: item.area)
                    
                    # Extraer propiedades. regionprops devuelve ejes en píxeles.
                    # Se multiplican por el espaciado para obtener milímetros físicos.
                    d_mayor_mm = prop_aorta.axis_major_length * pixel_spacing_mm
                    d_menor_mm = prop_aorta.axis_minor_length * pixel_spacing_mm
                    excentricidad = prop_aorta.eccentricity
                    
                    diametros_mayores.append(d_mayor_mm)
                    diametros_menores.append(d_menor_mm)
                    excentricidades.append(excentricidad)

        if not diametros_mayores:
            print(f" La máscara de {nombre_estructura} está completamente vacía.")
            return None

        # Calcular estadísticas
        resultados = {
            "diametro_mayor": {
                "max": np.max(diametros_mayores),
                "min": np.min(diametros_mayores),
                "promedio": np.mean(diametros_mayores),
                "std": np.std(diametros_mayores)
            },
            "diametro_menor": {
                "max": np.max(diametros_menores),
                "min": np.min(diametros_menores),
                "promedio": np.mean(diametros_menores),
                "std": np.std(diametros_menores)
            },
            "excentricidad": {
                "max": np.max(excentricidades),
                "min": np.min(excentricidades),
                "promedio": np.mean(excentricidades),
                "std": np.std(excentricidades)
            }
        }

        return (
            resultados['excentricidad']['max'], 
            resultados['excentricidad']['min'], 
            resultados['excentricidad']['promedio'], 
            resultados['excentricidad']['std'],
            resultados['diametro_mayor']['max'], 
            resultados['diametro_mayor']['min'], 
            resultados['diametro_mayor']['promedio'], 
            resultados['diametro_mayor']['std'],
            resultados['diametro_menor']['max'], 
            resultados['diametro_menor']['min'], 
            resultados['diametro_menor']['promedio'], 
            resultados['diametro_menor']['std']
        )

    vol_opt = calcular_volumen(mask_limpia_sitk, "Máscara Completa")
    vol_asc = calcular_volumen(mask_arco_asc, "Arco Ascendente")
    vol_des = calcular_volumen(mask_arco_desc, "Arco Descendente")
    vol_dsc = calcular_volumen(mask_descenso, "Descenso de la Aorta")

    area_mm2 = calculate_surface_area(mask_limpia_sitk, voxel_dims)

    # Para el arco ascendente
    asc_area_total_mm2 = calcular_areas_salida(mask_arco_asc, "Arco Ascendente", tipo_cara='primera')

    # Para el descenso
    desc_area_total_mm2 = calcular_areas_salida(mask_descenso, "Descenso", tipo_cara='primera')

    # Para el descenso
    (excent_max_desc, excent_min_desc, excent_prom_desc, excent_std_desc,
    major_diam_max_desc, major_diam_min_desc, major_diam_prom_desc, major_diam_std_desc,
    minor_diam_max_desc, minor_diam_min_desc, minor_diam_prom_desc, minor_diam_std_desc) = calcular_metricas_transversales(mask_descenso, "Descenso de la Aorta", voxel_dims)

    features_df = pd.DataFrame({
        "Study Name": file_name,
        "Condition": condition,
        "Volumen_Total_cm3": vol_opt,
        "Volumen_Ascendente_cm3": vol_asc,
        "Volumen_ArcoDescendente_cm3": vol_des,
        "Volumen_Descenso_cm3": vol_dsc,
        "Area_Superficie_Total_mm2": area_mm2,
        "Area_Salida_Ascendente_mm2": asc_area_total_mm2,
        "Area_Salida_Descenso_mm2": desc_area_total_mm2,
        "Excentricidad_Max_Desc": excent_max_desc,
        "Excentricidad_Min_Desc": excent_min_desc,
        "Excentricidad_Promedio_Desc": excent_prom_desc,
        "Excentricidad_Std_Desc": excent_std_desc,
        "Diametro_Mayor_Max_Desc_mm": major_diam_max_desc,
        "Diametro_Mayor_Min_Desc_mm": major_diam_min_desc,
        "Diametro_Mayor_Promedio_Desc_mm": major_diam_prom_desc,
        "Diametro_Mayor_Std_Desc_mm": major_diam_std_desc,
        "Diametro_Menor_Max_Desc_mm": minor_diam_max_desc,
        "Diametro_Menor_Min_Desc_mm": minor_diam_min_desc,
        "Diametro_Menor_Promedio_Desc_mm": minor_diam_prom_desc,
        "Diametro_Menor_Std_Desc_mm": minor_diam_std_desc
    }, index=[0])

    return features_df

all_features_df = []

for index, row in dataset.iterrows():
    condition = row["Etiqueta"]
    file_name = row["Nombre_Estudio"]
    input_path = dataset_path / condition / file_name / "aorta.nii.gz"
    print(f"Procesando {file_name}")
    study_df = features_extractor(input_path,file_name, condition)
    all_features_df.append(study_df)

full_df = pd.concat(all_features_df, ignore_index=False)
full_df.to_csv("dataset_features.csv", index=False)

Procesando Mario_D'Oria_ADX_ITA_GE_0.625
Procesando Mario_D'Oria_ADX_ITA_PHILIPS_1.000
Procesando Mario_D'Oria_ADX_ITA_PHILIPS_2.000
Procesando Daniele_Morosetti_ADX_ITA_GE_2.5
Procesando Daniele_Morosetti_ADX_ITA_GE_2.5 1
